# 🧪 PT-W4-D1 概念实验：Entity / Identity / Relationship 三层升级

> 配套阅读：`PT-W4-D1-升级OntologyModel.md`（现状盘点、ADR-006 四类关系、语义命名表在那边）
> 这个 notebook 把 MI CRE Semantic Model v0.1 的第一层建成**可运行的迷你版**，回答：
> 1. **Entity 层**：一句业务定义为什么是 Agent 的概念入口？
> 2. **Identity 层**：扁平编码 vs 层级路径 —— 删掉主键，业务还认得出 A101 吗？
> 3. **Relationship 层**：四类关系 × 动词注册制 —— 动词为什么是可推理的边，不是注释？
>
> 最后跑 md 版第五节的验证场景：**“A101 铺位为什么不能出租？”**（升级前查代码 vs 升级后遍历语义图）。
> 范式沿用 PT-W4-D2（明日课程）：类型层冻结 → 声明层治理 → 实例层运行。

## 第 1 层：Entity —— 从“表名+不变量”到“带业务定义的 Concept”

Domain Model v1.0 里 Resource Unit 只有 Owner Context 和不变量（编码全局唯一）——AI 只知道谁管它，不知道它在业务世界里**是什么**。
升级动作：每个 Concept 一句业务定义 + 入口关键词。定义不是给人看的注释，是 Agent 定位概念的索引。

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Concept:
    name: str
    definition: str            # 一句话业务定义（Agent 的概念入口）
    identity: str              # 业务身份：业务上怎么“认出”它
    entry_keywords: tuple      # 什么问题应该路由到这个概念

CONCEPTS = {
    "ResourceUnit": Concept(
        "ResourceUnit",
        "商场中可被独立出租、计价与运营的最小空间单元；它不是一行记录，是可出租能力的载体",
        "层级路径 Project/Building/Floor/Unit",
        ("铺位", "空间", "楼层", "面积")),
    "Occupancy": Concept(
        "Occupancy",
        "某个铺位在某段时间内被某份合同占住的事实；铺位是否可租由它决定，而不是由 status 字段决定",
        "（unit, contract, 时间段）三元组",
        ("出租", "可租", "占用", "空置")),       # ← 注意：能租/不能租的问题入口在 Occupancy
    "Merchant": Concept(
        "Merchant",
        "与商场建立合同关系的经营主体（B2B），不是消费者（Customer 是 B2C）",
        "名称 + 证件号",
        ("商户", "经营主体", "品牌")),
    "Contract": Concept(
        "Contract",
        "商场与商户之间关于铺位使用与计费的协议；业务事实的源头，所有 effect 由它的生命周期触发",
        "合同编号（全局唯一）",
        ("合同", "租约", "条款")),
}

print("Concept 卡片（升级后每个对象自带业务定义）：")
for c in CONCEPTS.values():
    print(f"  ▸ {c.name}")
    print(f"      定义: {c.definition}")
    print(f"      身份: {c.identity}")
print()

def locate_concepts(question: str) -> list[str]:
    """Agent 第一步：沿业务定义定位概念（升级前只能靠猜表名/查代码）"""
    return [name for name, c in CONCEPTS.items()
            if any(k in question for k in c.entry_keywords)]

q = "A101 为什么不能出租？"
hits = locate_concepts(q)
print(f"问题：{q!r}")
print(f"概念定位 → {hits}")
print("路由到 Occupancy（可租性归它管），而不是 ResourceUnit.status ——")
print("没有这句业务定义，Agent 读到“A101 不能出租”时根本不知道第一步该查哪个对象")

## 第 2 层：Identity —— 从扁平编码到「业务身份层级」

Identity 的 Ontology 判据：**删掉主键，业务还能不能认出它？**
- Merchant：名称+证件号 → 能 ✅（这就是业务身份）
- Resource Unit：只有编码 → 多项目下同名 A101 直接撞车 ❌

A101 的业务身份是一条路径：`Project → Building → Floor → Resource Unit`。用 dataclass 建两个项目验证撞车与解歧。

In [ ]:
from dataclasses import dataclass

@dataclass
class HNode:
    """层级包含（ADR-006 Hierarchical Containment）的实例"""
    obj_id: str
    obj_type: str
    parent: "HNode | None" = None

    def path(self) -> list[str]:
        return (self.parent.path() if self.parent else []) + [self.obj_id]

# 两个项目 —— 各自都有一个叫 A101 的铺位
hj = HNode("恒基广场", "Project")
l1a = HNode("L1", "Floor", HNode("T1", "Building", hj))
a101_hj = HNode("A101", "ResourceUnit", l1a)

xj = HNode("星际广场", "Project")
l1b = HNode("L1", "Floor", HNode("M1", "Building", xj))
a101_xj = HNode("A101", "ResourceUnit", l1b)

units = [a101_hj, a101_xj]

def resolve_by_code(code: str):
    return [u for u in units if u.obj_id == code]

def resolve_by_path(path: list[str]):
    return [u for u in units if u.path() == path]

print("扁平编码 resolve('A101')：")
for u in resolve_by_code("A101"):
    print("  命中:", "/".join(u.path()))
print("  → 2 个项目都有 A101：编码不构成业务身份（删掉主键，认不出是哪个）")
print()
print("层级路径 resolve(['星际广场','M1','L1','A101'])：")
for u in resolve_by_path(["星际广场", "M1", "L1", "A101"]):
    print("  命中:", "/".join(u.path()), f"(type={u.obj_type})")
print("  → 唯一定位。出租率/动线分析依赖的正是这条层级身份，而非扁平编码")
print()

# Merchant 的业务身份：名称+证件号（不是自增主键）
@dataclass(frozen=True)
class Merchant:
    name: str
    license_no: str

registry = {}
sbux = Merchant("星巴克", "91310000MA1K3X2Y")
registry[(sbux.name, sbux.license_no)] = sbux
found = registry[("星巴克", "91310000MA1K3X2Y")]
print(f"Merchant 业务身份检索：('星巴克','91310000MA1K3X2Y') → {found}")
print("数据库主键删了照样认出 —— 这才叫业务身份（Identity 判据通过）")
print()
print("结构红利：parent 链天然无环 → “层级不可循环”不变量挂在身份结构上，不挂在校验代码里")

## 第 3 层：Relationship —— 四类关系 × 动词注册制

ADR-006 冻结的四类关系是资产，**缺的是语义动词**。“合同↔铺位”（名词对名词）AI 无法推理；
`Contract occupies ResourceUnit` 才是可遍历的边。动词词表像 effect-registry 一样**注册制封闭**——
随手新增 `related-to` 这种通用关系会被治理拒绝（ADR-001 §7.2 否决过的老路）。

下面建迷你语义图：对象创建 → 关系绑定（动词必须在册）。

In [ ]:
from collections import defaultdict

# 类型层：四类关系（ADR-006 冻结）× 注册动词词表（封闭，ORE-1 同构）
RELATION_REGISTRY = {
    "Identity Reference":          {"signs-with"},
    "Structural Composition":      {"equipped-with"},
    "Hierarchical Containment":    {"contains"},
    "Lifecycle Transition Effect": {"occupies", "generates", "enables"},
}
VERB_CATEGORY = {v: cat for cat, verbs in RELATION_REGISTRY.items() for v in verbs}

# 实例层：语义图（src --verb--> dst）
graph = defaultdict(list)

def bind(verb: str, src: str, dst: str) -> str:
    """关系绑定：动词必须在注册词表内，否则治理拒绝"""
    if verb not in VERB_CATEGORY:
        raise ValueError(
            f"未注册动词 {verb!r} —— 词表封闭（注册制），"
            f"通用无类型关系是 ADR-001 §7.2 否决过的老路")
    graph[src].append((verb, dst))
    return f"{src} --{verb}--> {dst}"

# 对象创建 → 关系绑定（沿用第 2 层的层级身份）
for line in [
    bind("contains",     "恒基广场", "T1"),
    bind("contains",     "T1",       "L1"),
    bind("contains",     "L1",       "A101"),
    bind("signs-with",   "C-2026-0810", "星巴克"),        # Identity Reference
    bind("occupies",     "C-2026-0810", "A101"),          # Lifecycle Effect（occupancy）
    bind("generates",    "C-2026-0810", "BILL-2026-09"),  # Lifecycle Effect（financial）
    bind("equipped-with","A101",     "电表EM-01"),        # Structural Composition
]:
    print(line)

# 治理反例：想加一条随手命名的关系
try:
    bind("related-to", "C-2026-0810", "A101")
except ValueError as e:
    print("\n治理拒绝：", e)

print("\n按四类归类（动词不是注释，是带类别的边类型）：")
for cat, verbs in RELATION_REGISTRY.items():
    n = sum(1 for lst in graph.values() for v, _ in lst if v in verbs)
    print(f"  {cat:<26} {n} 条")

## 第 4 层：Agent 推理 —— “A101 为什么不能出租？”（md 版第五节验证场景）

升级前：查表 `resource WHERE code='A101'`，读 FK `lease.space_id`，语义靠猜，判断靠 if-else 代码。
升级后：**遍历语义图** —— ① Entity 定位概念 → ② 沿 `occupies` 反向找 Active 合同 → ③ 引用 Occupancy 定义作答。
再跑一个正向影响面分析：“C-2026-0810 终止后影响什么” —— 一句话自动展开为推理链。

In [ ]:
def reverse_lookup(verb: str, dst: str) -> list[str]:
    """沿动词边反向遍历（升级前：读 FK lease.space_id，语义靠猜）"""
    return [src for src, edges in graph.items() for v, d in edges if v == verb and d == dst]

def why_not_leasable(unit_id: str) -> None:
    print(f"查询：{unit_id} 为什么不能出租？\n")
    # ① Entity 层：概念定位
    print("① 概念定位：可租性由 Occupancy 定义决定（不是 ResourceUnit.status 字段）")
    # ② Relationship 层：沿 occupies 反向遍历
    contracts = reverse_lookup("occupies", unit_id)
    print(f"② 沿 `occupies` 反向遍历：{unit_id} ←--occupies-- {contracts}")
    for cid in contracts:
        # ③ 关联升级前要读 FK + 猜语义；现在动词就是语义
        print(f"③ Contract {cid} 存在 Active Occupancy（铺位被占住的事实）")
        print(f"④ 结论：{unit_id} 不可出租。若合同迁移到 Terminated → occupancy-effect → 铺位释放（明日 D2 的守卫与 Event）")

why_not_leasable("A101")

print("\n" + "─" * 56)
print("影响面分析：Contract C-2026-0810 终止后影响什么？（正向遍历出边）\n")
for verb, dst in graph["C-2026-0810"]:
    cat = VERB_CATEGORY[verb]
    if cat == "Lifecycle Transition Effect":
        print(f"  C-2026-0810 --{verb}--> {dst}")
        print(f"    ├ 类别: {cat}")
        print(f"    └ 推理: 合同终止 → 撤销该边 → {'铺位释放(可租)' if dst == 'A101' else '账单停止生成'}")

print("\n升级前 vs 升级后：")
print("  升级前: FK lease.space_id='A101' —— 占用？预留？共享？语义靠读代码")
print("  升级后: occupies/generates —— 动词即推理边，一条查询展开整条因果链")

## 第 5 层：可视化 —— 迷你语义图（动词即边，四类着色）

把第 3 层建好的图画出来：节点=Concept 实例，边=注册动词，颜色=四类关系。
这张图就是 Semantic Model v0.1 第一层的全部资产 —— 后面的 Rule/Policy/Capability 都挂在它上面。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

CAT_COLOR = {
    "Hierarchical Containment":   "#4c72b0",
    "Identity Reference":         "#dd8452",
    "Lifecycle Transition Effect": "#55a868",
    "Structural Composition":     "#c44e52",
}

pos = {
    "恒基广场": (0.06, 0.86), "T1": (0.28, 0.86), "L1": (0.48, 0.86),
    "A101": (0.68, 0.86), "电表EM-01": (0.90, 0.86),
    "星巴克": (0.18, 0.30), "C-2026-0810": (0.50, 0.55), "BILL-2026-09": (0.84, 0.38),
}
NODE_FC = {"C-2026-0810": "#d5e8d4", "星巴克": "#ffe6cc",
           "BILL-2026-09": "#dae8fc"}

fig, ax = plt.subplots(figsize=(11, 5.5))
for src, edges in graph.items():
    for verb, dst in edges:
        if src not in pos or dst not in pos:
            continue
        x1, y1 = pos[src]; x2, y2 = pos[dst]
        color = CAT_COLOR[VERB_CATEGORY[verb]]
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="->", color=color, lw=2.0,
                                    shrinkA=30, shrinkB=30))
        ax.text((x1 + x2) / 2 + 0.015, (y1 + y2) / 2 + 0.02, verb,
                fontsize=10, color=color, style="italic")

for name, (x, y) in pos.items():
    ax.text(x, y, name, fontsize=11, ha="center", va="center",
            bbox=dict(boxstyle="round,pad=0.45",
                      fc=NODE_FC.get(name, "#f5f5f5"), ec="#666666"))

handles = [Line2D([0], [0], color=c, lw=2.5, label=cat)
           for cat, c in CAT_COLOR.items()]
ax.legend(handles=handles, loc="lower left", fontsize=9, framealpha=0.9)
ax.set_xlim(0, 1); ax.set_ylim(0.15, 1.0)
ax.set_title("MI CRE Semantic Model v0.1 · 第一层迷你版：动词即推理边（四类封闭）", fontsize=12)
ax.axis("off")
plt.tight_layout()
plt.show()

print("读图：绿色（occupies/generates）是 Lifecycle 边 —— 明日 D2 在这些边上挂守卫与 Event；")
print("蓝色 contains 链就是 A101 的层级身份；橙色 signs-with 挂 Merchant 的名称+证件号身份")

## 结论：三层升级给 AI 的可读性

| 层 | 升级前（Domain Model） | 本实验 | 治理方式 |
|---|---|---|---|
| Entity | 表名 + Owner + 不变量 | `CONCEPTS`：一句业务定义 + 入口关键词 | 每个 Concept 定义必填 |
| Identity | 编码全局唯一（扁平） | 层级路径解析 + 名称/证件号检索 | 结构自带无环不变量 |
| Relationship | 名词对名词（合同↔铺位） | `RELATION_REGISTRY` 四类 × 动词注册制 | 词表封闭，随手新增被拒 |

**升级前 AI 需要“理解代码”；升级后 AI 只需要“遍历语义图”。**（第 4 层两个查询已验证）

> 练习（md 版第七节）：给 `Contract ↔ Deposit` 补一条关系 —— 属四类中的哪一类？动词怎么命名？
> 提示：先跑 `VERB_CATEGORY`，Deposit 的条款归 Contract、资金流归 Collection，别把两条塞进一个动词。

→ 深入阅读：同目录 `.md` 版本第二~四节（升级动作明细 + 语义命名全表 + 命名治理）
→ 明日 D2：升级 Lifecycle 与 Event —— 在 `occupies` 这些边上挂守卫、Event 与 Effect（配套实验 notebook 已就位）